<div style="border-top:4px solid #0f766e;padding:28px 0 18px">
<div style="color:#0f766e;font-size:13px;font-weight:700;letter-spacing:.8px">LAB 05 · LEVEL 2 · ANALYZING DATA</div>
<div style="color:#17212b;font-size:30px;font-weight:750">Turn event detail into explainable metrics</div>
<p style="color:#475569;font-size:15px;line-height:1.7;max-width:900px">Use filtering, conditional aggregation, CTEs, ranking, lag, and cumulative windows to answer business questions without losing track of result grain.</p>
<span style="display:inline-block;border:1px solid #99f6e4;background:#f0fdfa;color:#115e59;padding:6px 10px;margin-top:10px;font-size:12px">Prerequisite: Lab 4 · run cells in order</span>
</div>

## Contract for this lab

The input is the typed `modeling_typed_events` table created in Lab 4. Each row is one event. Every query below states its output grain before it runs, and the final checks are rendered as tables so that row counts and totals are visible.

In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "doris_course").is_dir()
)
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from doris_course import DorisLab

lab = DorisLab(lab_dir=COURSE_ROOT)
lab.connect(container="doris", host="127.0.0.1", port=9030)
lab.execute("USE doris_course")


In [ ]:
lab.sql("""
SELECT event_time, event_id, user_id, event_type, region, product_id, revenue
FROM modeling_typed_events
ORDER BY event_time, event_id
LIMIT 8
""", title="Typed event input")

## 1. Daily and regional rollup

**Result grain:** one row per event date and region. The CTE names the intermediate grain before the final ordering.

In [ ]:
lab.sql("""
WITH daily_region AS (
    SELECT
        TO_DATE(event_time) AS event_date,
        region,
        COUNT(*) AS event_count,
        COUNT(DISTINCT user_id) AS active_users,
        SUM(revenue) AS revenue
    FROM modeling_typed_events
    GROUP BY TO_DATE(event_time), region
)
SELECT event_date, region, event_count, active_users, revenue
FROM daily_region
ORDER BY event_date, region
""", title="Daily regional metrics")

## 2. Conditional aggregation

**Result grain:** one row per event date. Conditional expressions keep view, cart, and purchase metrics in one record.

In [ ]:
lab.sql("""
SELECT
    TO_DATE(event_time) AS event_date,
    SUM(CASE WHEN event_type = 'view' THEN 1 ELSE 0 END) AS view_events,
    SUM(CASE WHEN event_type = 'cart' THEN 1 ELSE 0 END) AS cart_events,
    SUM(CASE WHEN event_type = 'purchase' THEN 1 ELSE 0 END) AS purchase_events,
    SUM(CASE WHEN event_type = 'purchase' THEN revenue ELSE 0 END) AS purchase_revenue
FROM modeling_typed_events
GROUP BY TO_DATE(event_time)
ORDER BY event_date
""", title="Funnel metrics by day")

## 3. Rank products within each region

The aggregate CTE first creates one row per region and product. `ROW_NUMBER` then keeps the result grain at one row per product while adding a rank.

In [ ]:
lab.sql("""
WITH product_revenue AS (
    SELECT
        region,
        product_id,
        SUM(revenue) AS revenue
    FROM modeling_typed_events
    WHERE product_id IS NOT NULL
    GROUP BY region, product_id
), ranked AS (
    SELECT
        region,
        product_id,
        revenue,
        ROW_NUMBER() OVER (
            PARTITION BY region
            ORDER BY revenue DESC, product_id
        ) AS revenue_rank
    FROM product_revenue
)
SELECT region, product_id, revenue, revenue_rank
FROM ranked
WHERE revenue_rank <= 3
ORDER BY region, revenue_rank
""", title="Top products by region")

## 4. Compare each day with the previous day

`LAG` preserves one row per day. It does not collapse rows like `GROUP BY`; it adds the previous value to the existing grain.

In [ ]:
lab.sql("""
WITH daily AS (
    SELECT
        TO_DATE(event_time) AS event_date,
        SUM(revenue) AS revenue
    FROM modeling_typed_events
    GROUP BY TO_DATE(event_time)
)
SELECT
    event_date,
    revenue,
    LAG(revenue, 1, 0) OVER (ORDER BY event_date) AS previous_revenue,
    revenue - LAG(revenue, 1, 0) OVER (ORDER BY event_date) AS day_change,
    SUM(revenue) OVER (
        ORDER BY event_date
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS cumulative_revenue
FROM daily
ORDER BY event_date
""", title="Daily change and cumulative revenue", final=True)

## Takeaway

- Aggregates change the result grain; window functions enrich the current grain.
- A deterministic window order includes a tie-breaker when the business result requires one.
- A CTE is useful when the next step depends on a clearly named intermediate grain.